In [15]:
import pandas as pd
import numpy as np

from sqlalchemy import create_engine

from sklearn.metrics import mean_absolute_error, mean_squared_error
from xgboost import XGBRegressor

In [19]:
connection_string = (
    "mssql+pyodbc://@localhost/mlb?"
    "driver=ODBC+Driver+17+for+SQL+Server&trusted_connection=yes"
)

engine = create_engine(connection_string)

query = """
SELECT *
FROM mlb.dbo.fact_hitter_pitcher_matchup_model_features
"""

df = pd.read_sql(query, engine)

print(df.shape)
df.head()

(24641, 499)


,gamePk,game_date,season,hitter_id,hitter_name,hitter_position,hitter_team_id,hitter_team_name,pitcher_id,pitcher_name,...,pitcher_weighted_csw_rate_last_10,pitcher_weighted_sc_strike_rate_last_10,pitcher_weighted_velocity_last_10,pitcher_weighted_spin_rate_last_10,pitcher_weighted_chase_rate_last_10,pitcher_weighted_putaway_rate_last_10,pitcher_prev_whiff_rate,pitcher_prev_csw_rate,pitcher_prev_chase_rate,pitcher_strikeOuts
0,778068,2025-05-04,2025,516782,Starling Marte,LF,118,Kansas City Royals,669467,Andre Pallante,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
1,823401,2026-04-06,2026,687462,Spencer Horwitz,1B,134,Pittsburgh Pirates,608566,Germán Márquez,...,NaN,NaN,NaN,NaN,NaN,NaN,0.030769,0.261538,0.269231,4
2,824698,2026-04-01,2026,518595,Travis d'Arnaud,C,108,Los Angeles Angels,571510,Matthew Boyd,...,NaN,NaN,NaN,NaN,NaN,NaN,0.285714,0.412698,0.297297,10
3,777137,2025-07-12,2025,543309,Kyle Higashioka,C,140,Texas Rangers,664285,Framber Valdez,...,0.280093,0.465605,88.337834,2406.148003,0.332585,0.210342,0.078431,0.235294,0.294118,10
4,776969,2025-07-28,2025,605137,Josh Bell,1B,142,Minnesota Twins,664285,Framber Valdez,...,0.272823,0.466111,88.458581,2395.127776,0.308990,0.209552,0.096774,0.225806,0.250000,12


In [3]:
# to a list of all the columns in a txt file 
with open("columns.txt", "w") as f:
    for col in df.columns:
        f.write(col + "\n")

In [20]:
# define target and remove non-model columns
target = "pitcher_strikeOuts"

drop_cols = [
    "gamePk",
    "game_date",
    "hitter_name",
    "pitcher_name",
    "hitter_position",
    "hitter_team_name",
    "pitcher_team_name"
]

# same-game leakage columns: do not use for prediction
leakage_cols = [
    "hitter_strikeOuts",
    "pitches_seen_vs_pitcher",
    "swings_vs_pitcher",
    "whiffs_vs_pitcher",
    "called_strikes_vs_pitcher",
    "matchup_whiff_rate",
    "matchup_called_strike_rate",
    "matchup_csw_rate"
]

X = df.drop(columns=drop_cols + leakage_cols + [target], errors="ignore")
y = df[target]

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (24641, 483)
y shape: (24641,)


In [21]:
# choose 2.4 features
important_features = [
    # ===== Pitcher form (WEIGHTED) =====
    "pitcher_weighted_k_last_3",
    "pitcher_weighted_k_last_5",
    "pitcher_prev_k",

    # ===== Opportunity =====
    "pitcher_avg_bf_last_3",
    "pitcher_avg_ip_last_3",
    "pitcher_avg_pitches_last_3",
    "pitcher_avg_outs_last_3",
    "pitcher_gamesStarted",

    # ===== Opportunity (WEIGHTED) =====
    "pitcher_weighted_bf_last_3",
    "pitcher_weighted_outs_last_3",

    # ===== Skill =====
    "pitcher_weighted_whiff_rate_last_3",
    "pitcher_avg_velocity_last_3",
    "pitcher_avg_putaway_rate_last_3",

    # ===== Hitter tendency =====
    "hitter_avg_k_last_3",
    "hitter_avg_k_last_5",

    # ===== Context =====
    "pitcher_throws",
    "hitter_stand"
]

In [22]:
# validate feature names before selecting
missing_cols = [col for col in important_features if col not in X.columns]
print("Missing columns:", missing_cols)

available_features = [col for col in important_features if col in X.columns]
print("Available feature count:", len(available_features))
print(available_features)

Missing columns: []
Available feature count: 17
['pitcher_weighted_k_last_3', 'pitcher_weighted_k_last_5', 'pitcher_prev_k', 'pitcher_avg_bf_last_3', 'pitcher_avg_ip_last_3', 'pitcher_avg_pitches_last_3', 'pitcher_avg_outs_last_3', 'pitcher_gamesStarted', 'pitcher_weighted_bf_last_3', 'pitcher_weighted_outs_last_3', 'pitcher_weighted_whiff_rate_last_3', 'pitcher_avg_velocity_last_3', 'pitcher_avg_putaway_rate_last_3', 'hitter_avg_k_last_3', 'hitter_avg_k_last_5', 'pitcher_throws', 'hitter_stand']


In [23]:
# subset to available features
X = X[available_features].copy()
print(X.shape)
X.head()

(24641, 17)


,pitcher_weighted_k_last_3,pitcher_weighted_k_last_5,pitcher_prev_k,pitcher_avg_bf_last_3,pitcher_avg_ip_last_3,pitcher_avg_pitches_last_3,pitcher_avg_outs_last_3,pitcher_gamesStarted,pitcher_weighted_bf_last_3,pitcher_weighted_outs_last_3,pitcher_weighted_whiff_rate_last_3,pitcher_avg_velocity_last_3,pitcher_avg_putaway_rate_last_3,hitter_avg_k_last_3,hitter_avg_k_last_5,pitcher_throws,hitter_stand
0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN,NaN,NaN,NaN,NaN,0.000000,0.0,R,R
1,NaN,NaN,1.0,18.000000,3.000000,65.000000,9.0,1.0,NaN,NaN,0.121368,90.509430,0.192099,0.333333,0.8,R,L
2,NaN,NaN,7.0,17.000000,3.200000,63.000000,11.0,1.0,NaN,NaN,NaN,88.914286,0.375000,2.000000,2.0,L,R
3,5.9,5.733333,7.0,27.333333,6.333333,95.333333,19.0,1.0,27.3,18.6,0.084868,88.476073,0.178114,0.666667,0.4,L,R
4,6.4,6.400000,4.0,26.666667,6.333333,96.333333,19.0,1.0,26.8,19.5,0.105563,89.148543,0.210159,1.000000,0.8,L,R


In [24]:
# time-based split
train_mask = df["season"] == 2025
test_mask = df["season"] == 2026

X_train = X.loc[train_mask].copy()
X_test = X.loc[test_mask].copy()

y_train = y.loc[train_mask].copy()
y_test = y.loc[test_mask].copy()

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (20905, 17)
X_test: (3736, 17)
y_train: (20905,)
y_test: (3736,)


In [25]:
# encode categorical columns
categorical_cols = [col for col in ["pitcher_throws", "hitter_stand"] if col in X_train.columns]

X_train = pd.get_dummies(X_train, columns=categorical_cols, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols, drop_first=True)

X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

print("Encoded X_train:", X_train.shape)
print("Encoded X_test:", X_test.shape)

Encoded X_train: (20905, 17)
Encoded X_test: (3736, 17)


In [26]:
# fill nulls
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print(X_train.isna().sum().sum(), X_test.isna().sum().sum())

0 0


In [27]:
# train model
model = XGBRegressor(
    n_estimators=100,
    max_depth=5,
    learning_rate=0.1,
    random_state=42
)

model.fit(X_train, y_train)

,"objective objective: typing.Union[str, xgboost.sklearn._SklObjWProto, typing.Callable[[typing.Any, typing.Any], typing.Tuple[numpy.ndarray, numpy.ndarray]], NoneType]Specify the learning task and the corresponding learning objective or a customobjective function to be used.For custom objective, see :doc:`/tutorials/custom_metric_obj` and:ref:`custom-obj-metric` for more information, along with the end note forfunction signatures.",'reg:squarederror'
,"base_score base_score: typing.Union[float, typing.List[float], NoneType]The initial prediction score of all instances, global bias.",None
,booster,None
,"callbacks callbacks: typing.Optional[typing.List[xgboost.callback.TrainingCallback]]List of callback functions that are applied at end of each iteration.It is possible to use predefined callbacks by using:ref:`Callback API `... note:: States in callback are not preserved during training, which means callback objects can not be reused for multiple training sessions without reinitialization or deepcopy... code-block:: python for params in parameters_grid: # be sure to (re)initialize the callbacks before each run callbacks = [xgb.callback.LearningRateScheduler(custom_rates)] reg = xgboost.XGBRegressor(**params, callbacks=callbacks) reg.fit(X, y)",None
,colsample_bylevel colsample_bylevel: typing.Optional[float]Subsample ratio of columns for each level.,None
,colsample_bynode colsample_bynode: typing.Optional[float]Subsample ratio of columns for each split.,None
,colsample_bytree colsample_bytree: typing.Optional[float]Subsample ratio of columns when constructing each tree.,None
,"device device: typing.Optional[str].. versionadded:: 2.0.0Device ordinal, available options are `cpu`, `cuda`, and `gpu`.",None
,"early_stopping_rounds early_stopping_rounds: typing.Optional[int].. versionadded:: 1.6.0- Activates early stopping. Validation metric needs to improve at least once in every **early_stopping_rounds** round(s) to continue training. Requires at least one item in **eval_set** in :py:meth:`fit`.- If early stopping occurs, the model will have two additional attributes: :py:attr:`best_score` and :py:attr:`best_iteration`. These are used by the :py:meth:`predict` and :py:meth:`apply` methods to determine the optimal number of trees during inference. If users want to access the full model (including trees built after early stopping), they can specify the `iteration_range` in these inference methods. In addition, other utilities like model plotting can also use the entire model.- If you prefer to discard the trees after `best_iteration`, consider using the callback function :py:class:`xgboost.callback.EarlyStopping`.- If there's more than one item in **eval_set**, the last entry will be used for early stopping. If there's more than one metric in **eval_metric**, the last metric will be used for early stopping.",None
,enable_categorical enable_categorical: boolSee the same parameter of :py:class:`DMatrix` for details.,False
,"eval_metric eval_metric: typing.Union[str, typing.List[typing.Union[str, typing.Callable]], typing.Callable, NoneType].. versionadded:: 1.6.0Metric used for monitoring the training result and early stopping. It can be astring or list of strings as names of predefined metric in XGBoost (See:doc:`/parameter`), one of the metrics in :py:mod:`sklearn.metrics`, or anyother user defined metric that looks like `sklearn.metrics`.If custom objective is also provided, then custom metric should implement thecorresponding reverse link function.Unlike the `scoring` parameter commonly used in scikit-learn, when a callableobject is provided, it's assumed to be a cost function and by default XGBoostwill minimize the result during early stopping.For advanced usage on Early stopping like directly choosing to maximize insteadof minimize, see :py:obj:`xgboost.callback.EarlyStopping`.See :doc:`/tutorials/custom_metric_obj` and :ref:`custom-obj-metric` for moreinformation... code-block:: python from sklearn.datasets import load_diabetes

In [28]:
# predict
y_pred = model.predict(X_test)

print(y_pred[:10])

[4.8811817 5.1144686 4.8650837 4.4431086 1.4121932 2.3208263 4.890072
 5.023115  5.3336425 5.0811706]


In [29]:
# evaluate
mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("MAE:", mae)
print("RMSE:", rmse)

MAE: 1.7175430059432983
RMSE: 2.2044083124493077


In [30]:
# attach predictions back to rows
results = test_df.copy()
results["predicted_strikeouts"] = y_pred

results[[
    "gamePk",
    "pitcher_name",
    "pitcher_team_name",
    "pitcher_strikeOuts",
    "predicted_strikeouts"
]].head(20)

,gamePk,pitcher_name,pitcher_team_name,pitcher_strikeOuts,predicted_strikeouts
1,823401,Germán Márquez,San Diego Padres,4,4.881182
2,824698,Matthew Boyd,Chicago Cubs,10,5.114469
30,824537,Brandon Williamson,Cincinnati Reds,3,4.865084
52,823648,Zac Gallen,Arizona Diamondbacks,5,4.443109
56,825105,Osvaldo Bido,Atlanta Braves,1,1.412193
80,823974,Andrew Hoffmann,Arizona Diamondbacks,0,2.320826
87,822916,George Kirby,Seattle Mariners,4,4.890072
89,823078,Matthew Liberatore,St. Louis Cardinals,2,5.023115
99,822756,Foster Griffin,Washington Nationals,6,5.333642
111,823732,Joe Boyle,Tampa Bay Rays,9,5.081171


In [31]:
# aggregate to pitcher-game level
pitcher_preds = results.groupby(
    ["gamePk", "pitcher_name", "pitcher_team_name"]
).agg(
    actual_K=("pitcher_strikeOuts", "first"),
    predicted_K=("predicted_strikeouts", "mean")
).reset_index()

pitcher_preds.head(20)

,gamePk,pitcher_name,pitcher_team_name,actual_K,predicted_K
0,822753,Brad Lord,Washington Nationals,2,2.228493
1,822753,Cole Henry,Washington Nationals,3,1.167167
2,822753,Michael McGreevy,St. Louis Cardinals,1,4.547887
3,822753,Miles Mikolas,Washington Nationals,3,4.768797
4,822753,PJ Poulin,Washington Nationals,0,1.333856
5,822753,Riley O'Brien,St. Louis Cardinals,0,1.153689
6,822754,Cade Cavalli,Washington Nationals,3,4.235724
7,822754,Cole Henry,Washington Nationals,1,1.293416
8,822754,George Soriano,St. Louis Cardinals,3,1.350618
9,822754,Matthew Liberatore,St. Louis Cardinals,6,4.730661


In [32]:
# feature importance
feature_importance = pd.DataFrame({
    "feature": X_train.columns,
    "importance": model.feature_importances_
}).sort_values("importance", ascending=False)

feature_importance.head(20)

,feature,importance
7,pitcher_gamesStarted,0.826269
15,pitcher_throws_R,0.017308
1,pitcher_weighted_k_last_5,0.014461
4,pitcher_avg_ip_last_3,0.014277
11,pitcher_avg_velocity_last_3,0.013867
0,pitcher_weighted_k_last_3,0.013766
10,pitcher_weighted_whiff_rate_last_3,0.013133
5,pitcher_avg_pitches_last_3,0.013080
12,pitcher_avg_putaway_rate_last_3,0.012858
8,pitcher_weighted_bf_last_3,0.011501


In [36]:
# optional betting edge example
pitcher_preds["line"] = 5.5
pitcher_preds["edge"] = pitcher_preds["predicted_K"] - pitcher_preds["line"]
pitcher_preds["bet"] = np.where(pitcher_preds["edge"] > 0, "OVER", "UNDER")

pitcher_preds.head(100)

,gamePk,pitcher_name,pitcher_team_name,actual_K,predicted_K,line,edge,bet
0,822753,Brad Lord,Washington Nationals,2,2.228493,5.5,-3.271507,UNDER
1,822753,Cole Henry,Washington Nationals,3,1.167167,5.5,-4.332833,UNDER
2,822753,Michael McGreevy,St. Louis Cardinals,1,4.547887,5.5,-0.952113,UNDER
3,822753,Miles Mikolas,Washington Nationals,3,4.768797,5.5,-0.731203,UNDER
4,822753,PJ Poulin,Washington Nationals,0,1.333856,5.5,-4.166144,UNDER
...,...,...,...,...,...,...,...,...
95,822919,MacKenzie Gore,Texas Rangers,9,5.683356,5.5,0.183356,OVER
96,822919,Sam Moll,Cincinnati Reds,1,1.423148,5.5,-4.076852,UNDER
97,822920,Jalen Beeks,Texas Rangers,0,1.297019,5.5,-4.202981,UNDER
98,822920,Kumar Rocker,Texas Rangers,3,3.751495,5.5,-1.748505,UNDER


In [34]:
# strong_bets = pitcher_preds[abs(pitcher_preds["edge"]) >= 1].copy()
strong_bets.sort_values("edge", ascending=False).head(20)
strong_bets = pitcher_preds[abs(pitcher_preds["edge"]) >= 1].copy()
strong_bets.sort_values("edge", ascending=False).head(20)

NameError: name 'strong_bets' is not defined